[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andyrdt/puzzles/blob/main/05_2026/starter_notebook.ipynb)

# Monthly Algorithmic Challenge — May 2026: Count Unique Tokens

*Inspired by Callum McDougall's [ARENA Monthly Algorithmic Challenges](https://learn.arena.education/chapter1_transformer_interp/monthly_algorithmic/).*

## Task

Given a sequence of 10 tokens drawn from a vocabulary of 10 symbols, predict the number of distinct symbols. The released model hits 100% accuracy on every count from 1 through 10 on a held-out test set. Reverse-engineer the algorithm it learned.

## The model

| | |
|---|---|
| **Input symbols** | 10 (rendered as `a` – `j`) |
| **Sequence length** | 10 input tokens |
| **Layers** | 2 |
| **Heads per layer** | 4 |
| **`d_model`** | 32 |
| **Parameters** | 9,600 |
| **Vocab** | 22 tokens (10 input symbols + `BOS` + `ANS` + 10 count tokens `#1`–`#10`) |
| **Positional embeddings** | none |
| **Causal mask** | yes |

Attention-only: no MLPs, no LayerNorm, no positional embeddings.

```
token_embedding
    → attention layer 1 (4 heads, causal mask) + residual
    → attention layer 2 (4 heads, causal mask) + residual
    → linear unembed → logits
```

## Submission

Submit a Colab notebook describing the algorithm.

## Setup

In [ ]:
%pip install -q torch einops nnsight==0.6.3 huggingface_hub matplotlib

In [ ]:
import json, importlib
from pathlib import Path

import torch
import matplotlib.pyplot as plt
from nnsight import NNsight
from huggingface_hub import hf_hub_download

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

REPO_ID = "andyrdt/05_2026_puzzle_1"
model_py_path = hf_hub_download(REPO_ID, "model.py")
spec = importlib.util.spec_from_file_location("model", model_py_path)
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)
AttentionOnlyTransformer = model_module.AttentionOnlyTransformer

config_path  = hf_hub_download(REPO_ID, "config.json")
weights_path = hf_hub_download(REPO_ID, "model.pt")

config = json.loads(Path(config_path).read_text())
raw_model = AttentionOnlyTransformer.from_config(config["model"])
raw_model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
raw_model.eval().to(device)
model = NNsight(raw_model)

print(f"Model config: {config['model']}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Vocabulary

| Token id range | Role | Render |
|---|---|---|
| `0` – `9` | Input symbols | `a` – `j` |
| `10` | `BOS` | `BOS` |
| `11` | `ANS` | `ANS` |
| `12` – `21` | Count outputs | `#1` – `#10` |

Input sequence: `[BOS, t1, t2, ..., t10, ANS]` (length 12). The model predicts a count token `#k` at the position following `ANS`. Input symbols and count tokens have disjoint id ranges.

In [ ]:
NUM_SYMBOLS = config["vocab"]["num_symbols"]   # 10
SEQ_LEN     = config["vocab"]["seq_len"]       # 10
BOS         = NUM_SYMBOLS                       # 10
ANS         = NUM_SYMBOLS + 1                   # 11
COUNT_BASE  = NUM_SYMBOLS + 2                   # 12  (count k -> COUNT_BASE + k - 1)
VOCAB_SIZE  = NUM_SYMBOLS + 2 + SEQ_LEN         # 22

def symbol_to_token(s):
    if isinstance(s, str):
        return ord(s) - ord("a")
    return int(s)

def count_token(k):
    return COUNT_BASE + (k - 1)

def token_label(tok):
    if 0 <= tok < NUM_SYMBOLS:
        return chr(ord("a") + tok)
    if tok == BOS: return "BOS"
    if tok == ANS: return "ANS"
    if COUNT_BASE <= tok < COUNT_BASE + SEQ_LEN:
        return f"#{tok - COUNT_BASE + 1}"
    return f"?{tok}"

def encode(symbols):
    assert len(symbols) == SEQ_LEN, f"Need {SEQ_LEN} symbols, got {len(symbols)}"
    return [BOS] + [symbol_to_token(s) for s in symbols] + [ANS]

def decode_count_token(tok):
    return tok - COUNT_BASE + 1

syms = ['a', 'b', 'a', 'c', 'a', 'b', 'd', 'a', 'c', 'e']
tokens = encode(syms)
print("input symbols   :", syms)
print("input token ids :", tokens)
print("input labels    :", [token_label(t) for t in tokens])
print("true unique cnt :", len(set(syms)))

In [ ]:
def plot_attention(attn_patterns, token_labels, title="Attention"):
    if not isinstance(attn_patterns, list):
        attn_patterns = [attn_patterns]
    n_layers = len(attn_patterns)
    n_heads = attn_patterns[0].shape[0]
    fig, axes = plt.subplots(n_layers, n_heads, figsize=(3.4 * n_heads, 3.0 * n_layers),
                             squeeze=False)
    for L, attn in enumerate(attn_patterns):
        attn = attn.detach().cpu().numpy()
        for h in range(n_heads):
            ax = axes[L][h]
            ax.imshow(attn[h], cmap="Blues", vmin=0, vmax=1)
            ax.set_xticks(range(len(token_labels)))
            ax.set_xticklabels(token_labels, rotation=45, ha="right",
                               rotation_mode="anchor", fontsize=8)
            ax.set_yticks(range(len(token_labels)))
            ax.set_yticklabels(token_labels, fontsize=8)
            ax.set_title(f"Layer {L}, Head {h}", fontsize=10)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## Running the model with `nnsight`

[`nnsight`](https://nnsight.net/) lets you capture activations and run interventions inside a `with model.trace(x): ...` block:

```python
with model.trace(x):
    out = model.output.save()
logits, attn_patterns = out
```

Intermediate activations work the same way (e.g. `model.layers[0].output.save()`); interventions are in-place mutations of activation tensors inside the block.

## Example outputs

In [ ]:
examples = [
    ['a','a','a','a','a','a','a','a','a','a'],   # 1 unique
    ['a','b','a','b','a','b','a','b','a','b'],   # 2 unique
    ['a','b','c','a','b','c','a','b','c','a'],   # 3 unique
    ['a','b','c','d','e','a','b','c','d','e'],   # 5 unique
    ['a','b','c','d','e','f','g','a','b','c'],   # 7 unique
    ['a','b','c','d','e','f','g','h','i','j'],   # 10 unique
]

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, syms in zip(axes.flat, examples):
    x = torch.tensor([encode(syms)], device=device)

    with model.trace(x):
        out = model.output.save()
    logits, _ = out

    count_logits = logits[0, -1, COUNT_BASE:COUNT_BASE + SEQ_LEN]
    probs = torch.softmax(count_logits, dim=-1).detach().cpu()
    pred_count = int(probs.argmax().item()) + 1
    true_count = len(set(syms))

    colors = ["green" if k + 1 == true_count else "lightgray" for k in range(SEQ_LEN)]
    ax.bar(range(1, SEQ_LEN + 1), probs.numpy(), color=colors)
    ax.set_xticks(range(1, SEQ_LEN + 1))
    ax.set_xlabel("Predicted unique count")
    ax.set_ylabel("P(count)")
    ax.set_ylim(0, 1.05)
    status = "✓" if pred_count == true_count else "✗"
    ax.set_title(f"{''.join(syms)}\npred={pred_count}, true={true_count} {status}", fontsize=10)

plt.suptitle("Output distribution at ANS (green = correct count)", fontsize=13)
plt.tight_layout()
plt.show()

## Attention patterns

In [ ]:
syms = ['a','b','a','c','a','b','d','a','c','e']  # 5 unique: {a,b,c,d,e}
tokens = encode(syms)
x = torch.tensor([tokens], device=device)

with model.trace(x):
    out = model.output.save()
_, attn_patterns = out
labels = [token_label(t) for t in tokens]

plot_attention(
    [attn_patterns[0][0], attn_patterns[1][0]],
    labels,
    title=f"Attention on {''.join(syms)} (true unique count = {len(set(syms))})",
)

## Accuracy across all unique counts

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

n_per_count = 200
per_count_correct = {c: 0 for c in range(1, SEQ_LEN + 1)}
per_count_total   = {c: 0 for c in range(1, SEQ_LEN + 1)}

for c in range(1, SEQ_LEN + 1):
    for _ in range(n_per_count):
        symbols = rng.choice(NUM_SYMBOLS, size=c, replace=False)
        extras = rng.choice(symbols, size=SEQ_LEN - c, replace=True)
        seq = np.concatenate([symbols, extras])
        rng.shuffle(seq)
        x = torch.tensor([[BOS] + seq.tolist() + [ANS]], device=device)
        with model.trace(x):
            out = model.output.save()
        pred = decode_count_token(int(out[0][0, -1].argmax().item()))
        per_count_correct[c] += int(pred == c)
        per_count_total[c]   += 1

for c in range(1, SEQ_LEN + 1):
    acc = per_count_correct[c] / per_count_total[c]
    print(f"  unique count = {c:2d}:  {per_count_correct[c]:4d}/{per_count_total[c]:4d}  acc = {acc:.4f}")

## Weights

Direct access to every weight matrix via `raw_model`.

In [ ]:
for name, p in raw_model.named_parameters():
    print(f"  {name:40s}  shape = {tuple(p.shape)}")

## Your turn

- `model` — `nnsight`-wrapped; use `with model.trace(x): ...` to capture or intervene.
- `raw_model` — plain `nn.Module`; weights accessible directly.